In [5]:
import os
import json
import evaluate
from datasets import load_dataset

dataset_name = "multilex_tiny"

d_map = {
    "multilex_tiny": "summary/tiny",
    "multilex_short": "summary/short",
    "multilex_long": "summary/long",
    "eurlexsum": "reference",
    "eurlexsum_test": "reference",
    "eurlexsum_validation": "reference"
}

split = None
if "multilex" in dataset_name:
    dataset = load_dataset("allenai/multi_lexsum", name="v20230518")
    dataset = dataset["test"].filter(lambda x: x[d_map[dataset_name]] != None)[d_map[dataset_name]]
else:
    dataset = load_dataset("dennlinger/eur-lex-sum", "english")
    match dataset_name:
        case "eurlexsum_test":
            split = slice(len(dataset["train"]["summary"]), len(dataset["train"]["summary"] + dataset["test"]["summary"]))
            dataset = dataset["test"]["summary"]
        case "eurlexsum_validation":
            split = slice(len(dataset["train"]["summary"] + dataset["test"]["summary"]), len(dataset["train"]["summary"] + dataset["test"]["summary"] + dataset["validation"]["summary"]))
            dataset = dataset["validation"]["summary"]
        case "eurlexsum":
            split = slice(0,len(dataset["train"]["summary"] + dataset["test"]["summary"] + dataset["validation"]["summary"]))
            dataset = dataset["train"]["summary"] + dataset["test"]["summary"] + dataset["validation"]["summary"]

    print(split, split.stop - split.start)    
    # dataset = dataset["train"]["summary"] + dataset["test"]["summary"] + dataset["validation"]["summary"]

/home/keddie/anaconda3/envs/facilex_caselaw/lib/python3.11/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/home/keddie/anaconda3/envs/facilex_caselaw/lib/python3.11/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


In [6]:
rouge_scoring = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

import os
import json
def load_predicted_data(path):
    predicted = []
    ordered_files = os.listdir(path)
    ordered_files = sorted(ordered_files, key = lambda x: int(x.split(".")[0]))
    for file in ordered_files:
        predicted.append(json.load(open(path+file, "r"))[-1]["content"])
        print(path, file) if len(json.load(open(path+file, "r"))[-1]["content"]) < 1 else 1

    return predicted


In [7]:
from tqdm import tqdm
import numpy as np

def evaluation(path_data, results, model, prompt_type, selection_type, limit):
    if "eurlexsum" in dataset_name:
        predicted = load_predicted_data(path_data)[split][:limit]
    else:
        predicted = load_predicted_data(path_data)[:limit]

    print(len(predicted))
    r_scores = rouge_scoring.compute(predictions=predicted, references=dataset[:len(predicted)], use_stemmer = True)
    bert_scores = bertscore.compute(predictions=predicted, references=dataset[:len(predicted)], model_type="microsoft/deberta-xlarge-mnli", batch_size = 1, verbose=True, device=device)
    # mover_score = compute_mover_score(predictions=predicted, references=dataset[:len(predicted)], orig_ref=dataset, orig_pred=predicted)

    # mover_score = np.mean(mover_score)
    mover_score = 0
    bert_scores = {"f1": np.mean(bert_scores["f1"])}
    r_scores = {metric: round(np.mean(val), 3) for metric, val in r_scores.items()}

    # mover_score = mover/len(predicted)
    # bert_scores = {"f1": bs/len(predicted)}
    # chosen deberta lange due to https://github.com/Tiiiger/bert_score/blob/master/README.md and paper
    # for some reason the data is all loaded onto the GPU so it doesn't fit anymore
    # bs = 0
    # mover = 0
    # batch_size = 50
    # for pred_idx in tqdm(range(0,len(predicted),batch_size)):
    #     # in bertscore.py, in evaluate metric bert (from evaluate huggingface), added del of scorer model to reduce VRAM usage
    #     bs_aux = bertscore.compute(predictions=predicted[pred_idx:pred_idx+batch_size], references=dataset[pred_idx:pred_idx+batch_size], model_type="microsoft/deberta-xlarge-mnli", batch_size = 1)
    #     mover_aux = compute_mover_score(references=dataset[pred_idx:pred_idx+batch_size], predictions=predicted[pred_idx:pred_idx+batch_size], orig_ref = dataset, orig_pred=predicted)
    #     mover += sum(mover_aux)
    #     bs += sum(bs_aux["f1"])

    results["model"] += [model] * 5
    results["selection_type"] += [selection_type] * 5
    results["prompt_type"] += [prompt_type] * 5
    results["score_value"] += [r_scores["rouge1"], r_scores["rouge2"], r_scores["rougeL"], bert_scores["f1"], mover_score]
    results["score_type"] += ["rouge1", "rouge2", "rougeL", "bert_score", "mover_score"]

results = {"model": [], "selection_type": [], "prompt_type": [], "score_value": [], "score_type": []}
if "eurlexsum" in dataset_name:
    dataset_name_aux = dataset_name.split("_")[0]
    path_dir = f"answers/{dataset_name_aux}"
else:
    path_dir = f"answers/{dataset_name}"
iter = 0
for model in tqdm(os.listdir(f"{path_dir}/")):
    for prompt_type in os.listdir(f"{path_dir}/{model}/"):
        for selection_type in os.listdir(f"{path_dir}/{model}/{prompt_type}/"):
            path_data = f"{path_dir}/{model}/{prompt_type}/{selection_type}/"
            # evaluation(path_data, results, model, prompt_type, selection_type, len(dataset))
            iter += 1
            # print(f"Iteration {iter}/{4*3*4}")

100%|██████████| 3/3 [00:00<00:00, 9390.23it/s]


In [8]:
path_data

'answers/multilex_tiny/Meta-Llama-3-8B-Instruct/basic/first5last5_textrank/'

In [9]:
p = f"answers/{dataset_name}/Meta-Llama-3-8B-Instruct/cod/random_selection_textrank/"

In [10]:
# import re
# reg = r"[*]{0,}[Ss]ummary[ 0-9]+(?:\([0-9]{,3}[ a-z]+\)){0,}[*]{0,}|[\w ]+summary:"

In [11]:
l = []
ordered_files = os.listdir(p)
ordered_files = sorted(ordered_files, key = lambda x: int(x.split(".")[0]))

texts = []
for file in ordered_files:
    text = json.load(open(p + file, "r"))[-1]["content"]
    texts += [text]

In [12]:
import tiktoken
tokenizer = tiktoken.encoding_for_model("gpt-3.5")

len(tokenizer.encode(texts[0]))

44

In [19]:
texts[0]

"Here is the final and best summary:\n\nThe City of Montgomery, Alabama, was sued by several plaintiffs who claimed that the city's policies and practices violated their due process and equal protection rights. The plaintiffs alleged that they were"

In [20]:
dataset[0]

'People who were imprisoned for failing to pay traffic fines filed a lawsuit against the City of Montgomery(M.D. Ala.)'

In [13]:
r_scores = rouge_scoring.compute(predictions=texts, references=dataset[:len(texts)], use_stemmer = True)
r_scores str.replace()

{'rouge1': 0.17087854552195075,
 'rouge2': 0.03481832400278116,
 'rougeL': 0.12543663971326424,
 'rougeLsum': 0.1309553022259884}

In [14]:
import pickle
import pandas as pd

df = pd.DataFrame(pickle.load(open("results_multilex_tiny_cod_new.pickle", "rb")))
df

,model,selection_type,prompt_type,score_value,score_type
0,Mistral-7B-Instruct-v0.3,first5last5_bert,cod,0.157000,rouge1
1,Mistral-7B-Instruct-v0.3,first5last5_bert,cod,0.040000,rouge2
2,Mistral-7B-Instruct-v0.3,first5last5_bert,cod,0.123000,rougeL
3,Mistral-7B-Instruct-v0.3,first5last5_bert,cod,0.551285,bert_score
4,Mistral-7B-Instruct-v0.3,first5last5_bert,cod,0.000000,mover_score
...,...,...,...,...,...
175,Meta-Llama-3-8B-Instruct,first5last5_textrank,basic,0.193000,rouge1
176,Meta-Llama-3-8B-Instruct,first5last5_textrank,basic,0.044000,rouge2
177,Meta-Llama-3-8B-Instruct,first5last5_textrank,basic,0.141000,rougeL
178,Meta-Llama-3-8B-Instruct,first5last5_textrank,basic,0.567513,bert_score


In [15]:
df[df["model"] == "Meta-Llama-3-8B-Instruct"]

,model,selection_type,prompt_type,score_value,score_type
120,Meta-Llama-3-8B-Instruct,first5last5_bert,cod,0.181000,rouge1
121,Meta-Llama-3-8B-Instruct,first5last5_bert,cod,0.038000,rouge2
122,Meta-Llama-3-8B-Instruct,first5last5_bert,cod,0.135000,rougeL
123,Meta-Llama-3-8B-Instruct,first5last5_bert,cod,0.557367,bert_score
124,Meta-Llama-3-8B-Instruct,first5last5_bert,cod,0.000000,mover_score
125,Meta-Llama-3-8B-Instruct,random_selection_bert,cod,0.178000,rouge1
126,Meta-Llama-3-8B-Instruct,random_selection_bert,cod,0.039000,rouge2
127,Meta-Llama-3-8B-Instruct,random_selection_bert,cod,0.134000,rougeL
128,Meta-Llama-3-8B-Instruct,random_selection_bert,cod,0.556413,bert_score
129,Meta-Llama-3-8B-Instruct,random_selection_bert,cod,0.000000,mover_score
